# AFIR for a molecule on a surface with NVIDIA ALCHEMI, on a platform GPU

Places a molecule at random over a slab many times and relaxes every placement with MACE through the
[NVIDIA ALCHEMI toolkit](https://github.com/NVIDIA/nvalchemi-toolkit), in batches on one GPU: first
under an artificial force between molecule and slab, as in the artificial force induced reaction (AFIR) method,
then without it. The final structures are ranked by energy, and the lowest are saved as materials.

The workflow has three steps, each in its own Python environment: *Create Initial Configurations* places the
molecule over the slab with made, from the job's two materials, *Relax under AFIR Force (ALCHEMI)* relaxes the
configurations, and *Save Final Structures* saves the lowest as the job's final structures. This notebook
builds the slab, fetches the molecule, saves both as materials, creates the workflow, runs one job on the two
materials and shows the ranked configurations and the lowest final structures.

Outside the platform's JupyterLite, use Python 3.10 or 3.11 and install the packages once with `pip install "mat3ra-notebooks-utils[all]"`.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime

from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None  # set to use an organization as the owner, otherwise the personal account

# 3. Material parameters
CRYSTAL_NAME = "Cu"  # from Materials Standata
MILLER_INDICES = (1, 1, 1)
NUMBER_OF_LAYERS = 2
XY_SUPERCELL_MATRIX = [[4, 0], [0, 4]]
USE_CONVENTIONAL_CELL = False  # the supercell repeats the primitive surface cell
VACUUM = 15.0  # in angstroms
MOLECULE_NAME = "water"  # from PubChem

# 4. Workflow parameters
NUMBER_OF_CONFIGURATIONS = 100  # random placements of the molecule over the slab
GAMMA = 100  # kJ/mol, model collision energy of the AFIR force
RANDOM_SEED = 0  # of the random placements
NUMBER_OF_FINAL_STRUCTURES = 1  # the lowest-energy final structures kept and saved as materials
BATCH_SIZE = 100  # configurations relaxed together on the GPU
PYTHON_VERSION = "3.11.10"
PYTHON_BUILD = "Default"

# 5. Compute parameters
CLUSTER_NAME = "cluster-003"
QUEUE_NAME = QueueName.GOF  # H100 nodes on cluster-003 (GOF on-demand, GSF spot)
TIME_LIMIT = "01:00:00"

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()

### 2.3. Select account

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Build the slab

In [ ]:
from mat3ra.made.material import Material
from mat3ra.made.tools.helpers import create_slab
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.standata.materials import Materials

crystal = Material.create(Materials.get_by_name_first_match(CRYSTAL_NAME))
slab = create_slab(
    crystal,
    miller_indices=MILLER_INDICES,
    number_of_layers=NUMBER_OF_LAYERS,
    vacuum=VACUUM,
    xy_supercell_matrix=XY_SUPERCELL_MATRIX,
    use_conventional_cell=USE_CONVENTIONAL_CELL,
)
visualize(slab)

### 3.2. Fetch the molecule from PubChem

In [ ]:
import io
from urllib.parse import quote

from ase.io import read
from mat3ra.made.tools.convert import from_ase
from mat3ra.notebooks_utils.io import read_from_url

PUBCHEM_STRUCTURE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/SDF?record_type=3d"

molecule_sdf = await read_from_url(PUBCHEM_STRUCTURE_URL.format(name=quote(MOLECULE_NAME)))
molecule = Material.create(from_ase(read(io.StringIO(molecule_sdf), format="sdf")))
molecule.name = MOLECULE_NAME
visualize(molecule)

### 3.3. Save materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in (slab, molecule)]

## 4. Create the workflow
### 4.1. Step 1: Create Initial Configurations

Places the molecule `NUMBER_OF_CONFIGURATIONS` times over the slab, each at a random orientation and a random point
of the surface cell, with its lowest atom above the top slab atom by the sum of their covalent radii plus 0.5 Å, as
GRRM's random initial structures [1]. It writes them to `configurations.extxyz`, slab atoms tagged 0 and molecule
atoms 1, each with its index, rotation and position.

### 4.2. Step 2: Relax under AFIR Force (ALCHEMI)

Relaxes the configurations of `configurations.extxyz` in batches of `BATCH_SIZE` with the slab fixed [3], first under
the AFIR force [1] of strength `GAMMA` (100 kJ/mol with random orientations [2]) between molecule and slab, then
without it. It writes `ranked_configurations.csv` with every configuration ranked by final energy, its starting
orientation and the fragments its molecule ends in; `final_structures.extxyz` with the `NUMBER_OF_FINAL_STRUCTURES`
lowest; and `final_energies.png`, the final energy of each configuration. Another backend or another ALCHEMI version
replaces `afir.py` and `afir_requirements.txt`; steps 1 and 3 stay.

### 4.3. Step 3: Save Final Structures

Takes the lowest final structures, each with the configuration it started from, and saves them as the job's final
structures: each is a material, shown in this step's Results.

Each step's script and requirements are files in `afir_molecule_surface_alchemi/` beside this notebook, under the
names the workflow gives them on the platform; the cell below reads them. Each step runs in its own Python environment
on the cluster, installed from its requirements the first time and reused afterwards.

- `configurations_requirements.txt` and `final_structures_requirements.txt`: steps 1 and 3 (*Create Initial
  Configurations*, *Save Final Structures*), the made library, which builds the configurations and converts the
  structures.
- `afir_requirements.txt`: step 2 (*Relax under AFIR Force (ALCHEMI)*), NVIDIA ALCHEMI with MACE on the GPU. **ALCHEMI
  here is a pre-release: version 0.3.0 installed from GitHub commit
  [6189164](https://github.com/NVIDIA/nvalchemi-toolkit/commit/618916442783c152cd53b919164f07f80286f3da), not a PyPI
  release.** PyTorch is the CUDA 12.8 build and Warp the CUDA 12 build (the `--find-links` lines). Building this
  environment the first time takes about 10 minutes (measured on cluster-003).

In [ ]:
from pathlib import Path

STEP_FILES_DIRECTORY = Path("afir_molecule_surface_alchemi")
CONFIGURATIONS_SCRIPT = (STEP_FILES_DIRECTORY / "configurations.py").read_text()
CONFIGURATIONS_REQUIREMENTS = (STEP_FILES_DIRECTORY / "configurations_requirements.txt").read_text()
AFIR_SCRIPT = (STEP_FILES_DIRECTORY / "afir.py").read_text()
AFIR_REQUIREMENTS = (STEP_FILES_DIRECTORY / "afir_requirements.txt").read_text()
FINAL_STRUCTURES_SCRIPT = (STEP_FILES_DIRECTORY / "final_structures.py").read_text()
FINAL_STRUCTURES_REQUIREMENTS = (STEP_FILES_DIRECTORY / "final_structures_requirements.txt").read_text()

### 4.4. Configure the steps

When the job is created, the platform writes the job's materials into step 1's script (`materials`; the
non-periodic one is the molecule). Step 1 writes the configurations to `configurations.extxyz` in the
job's work folder, where steps 2 and 3 read them. Each step's files are named after it.

In [ ]:
import copy
import re

from mat3ra.notebooks_utils.core.entity.workflow.api import set_execution_unit_input
from mat3ra.standata.workflows import WorkflowStandata


def set_constants(script, constants):
    """Sets each named constant of a script to its value."""
    for name, value in constants.items():
        script = re.sub(rf"^({name} = )\S+", rf"\g<1>{value}", script, flags=re.M)
    return script


def configure_python_step(subworkflow, template_unit_name, subworkflow_name, file_name, script, requirements, results,
                          is_rendered_at_run_time):
    """
    Makes a Standata python subworkflow one step of the workflow.

    Args:
        subworkflow (dict): The subworkflow config, edited in place.
        template_unit_name (str): The name of the template's python unit.
        subworkflow_name (str): The step's name in the workflow, also the name of its python unit.
        file_name (str): The base name of the unit's script and requirements files.
        script (str): The unit's script.
        requirements (str): The packages installed for the unit, as a requirements.txt.
        results (list): The results the unit declares.
        is_rendered_at_run_time (bool): True to leave the script's placeholders (JOB_WORK_DIR, JOB_ID, UNIT_ID) to the
            compute node, which fills them when the step runs; False to let the platform fill them, with the job's
            materials, when the job is created.
    """
    unit = next(unit for unit in subworkflow["units"] if unit["name"] == template_unit_name)
    unit.update(name=subworkflow_name, head=True, results=results)
    for application in (subworkflow["application"], unit["application"]):
        application.update(version=PYTHON_VERSION, build=PYTHON_BUILD)
    set_execution_unit_input(unit, "script.py", script)
    set_execution_unit_input(unit, "requirements.txt", requirements)
    inputs = {unit_input["template"]["name"]: unit_input for unit_input in unit["input"]}
    inputs["script.py"]["isManuallyChanged"] = is_rendered_at_run_time
    inputs["script.py"]["template"]["name"] = f"{file_name}.py"
    inputs["requirements.txt"]["template"]["name"] = f"{file_name}_requirements.txt"
    subworkflow.update(name=subworkflow_name, units=[unit])


python_workflows = WorkflowStandata.filter_by_application("python")
workflow_config = copy.deepcopy(python_workflows.get_by_name_first_match("custom_script.json"))
workflow_config.update(name="Molecule on Surface: AFIR Search (ALCHEMI)", isMultiMaterial=True)
workflow_config["units"][0]["name"] = "Create Initial Configurations"
python_script_subworkflow = python_workflows.get_by_name_first_match("python_script.json")["subworkflows"][0]
afir_subworkflow_config = copy.deepcopy(python_script_subworkflow)
final_structures_subworkflow_config = copy.deepcopy(python_script_subworkflow)
final_structures_subworkflow_config.pop("_id")
next(unit for unit in final_structures_subworkflow_config["units"] if unit["name"] == "python").pop("flowchartId")

configure_python_step(
    workflow_config["subworkflows"][0], template_unit_name="custom_script",
    subworkflow_name="Create Initial Configurations", file_name="configurations",
    script=set_constants(CONFIGURATIONS_SCRIPT, {"NUMBER_OF_CONFIGURATIONS": NUMBER_OF_CONFIGURATIONS,
                                                 "RANDOM_SEED": RANDOM_SEED}),
    requirements=CONFIGURATIONS_REQUIREMENTS, results=[], is_rendered_at_run_time=False,
)
configure_python_step(
    afir_subworkflow_config, template_unit_name="python", subworkflow_name="Relax under AFIR Force (ALCHEMI)",
    file_name="afir",
    script=set_constants(AFIR_SCRIPT, {"GAMMA": GAMMA, "NUMBER_OF_FINAL_STRUCTURES": NUMBER_OF_FINAL_STRUCTURES,
                                       "BATCH_SIZE": BATCH_SIZE}),
    requirements=AFIR_REQUIREMENTS,
    results=[
        {"name": "file_content", "filetype": "csv", "basename": "ranked_configurations.csv"},
        {"name": "file_content", "filetype": "image", "basename": "final_energies.png"},
    ],
    is_rendered_at_run_time=True,
)
configure_python_step(
    final_structures_subworkflow_config, template_unit_name="python", subworkflow_name="Save Final Structures",
    file_name="final_structures",
    script=FINAL_STRUCTURES_SCRIPT, requirements=FINAL_STRUCTURES_REQUIREMENTS,
    results=[{"name": "final_structure"}],
    is_rendered_at_run_time=True,
)

### 4.5. Create and save the workflow

Step 3 ends with a loop that saves one final structure per pass, with its starting configuration as the initial
structure.

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.wode import Subworkflow, Workflow
from mat3ra.wode.units import Unit

endpoint_options = {"method": "POST", "headers": {"Content-Type": "application/json"},
                    "data": "{{ FINAL_STRUCTURE_REQUESTS[FINAL_STRUCTURE_INDEX] }}"}
save_unit = Unit(type="io", subtype="output", source="api", enableRender=True, name="Save final structure",
                 input=[{"type": "api", "endpoint": "structures", "endpoint_options": endpoint_options,
                         "name": "FINAL_STRUCTURE"}])
exit_unit = Unit(type="assignment", name="Count saved structures", operand="NUMBER_OF_SAVED_STRUCTURES",
                 value="FINAL_STRUCTURE_INDEX", input=[])
loop_units = [
    Unit(type="assignment", name="Init structure index", operand="FINAL_STRUCTURE_INDEX", value=0, input=[]),
    save_unit,
    Unit(type="assignment", name="Increment structure index", operand="FINAL_STRUCTURE_INDEX",
         value="FINAL_STRUCTURE_INDEX + 1", input=[]),
    Unit(**{"type": "condition", "name": "Check all structures saved",
            "statement": "FINAL_STRUCTURE_INDEX >= len(FINAL_STRUCTURE_REQUESTS)",
            "then": exit_unit.flowchartId, "else": save_unit.flowchartId, "maxOccurrences": NUMBER_OF_CONFIGURATIONS,
            "input": [{"name": "FINAL_STRUCTURE_INDEX", "scope": "global"},
                      {"name": "FINAL_STRUCTURE_REQUESTS", "scope": "global"}]}),
    exit_unit,
]
final_structures_subworkflow = Subworkflow(**final_structures_subworkflow_config)
for unit in loop_units:
    final_structures_subworkflow.add_unit(unit)
workflow = Workflow.create(workflow_config)
workflow.add_subworkflow(Subworkflow(**afir_subworkflow_config))
workflow.add_subworkflow(final_structures_subworkflow)
visualize_workflow(workflow)

saved_workflow = client.workflows.create(workflow.to_dict_without_special_keys(), owner_id=ACCOUNT_ID)
print(f"✅ Workflow saved to the collection: {saved_workflow['_id']}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[cluster['hostname'] for cluster in clusters]}")

### 5.2. Create compute configuration for the job

In [ ]:
from mat3ra.ide.compute import Compute

cluster = next(cluster for cluster in clusters if CLUSTER_NAME in cluster["hostname"])
compute = Compute(cluster=cluster, queue=QUEUE_NAME, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, nodes: {compute.nodes}, ppn: {compute.ppn}, time limit: {TIME_LIMIT}")

## 6. Create the job

In [ ]:
from mat3ra.notebooks_utils.job import create_job

job = create_job(
    api_client=client,
    materials=saved_materials,
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=f"{workflow.name} {MOLECULE_NAME} on {slab.name}, {NUMBER_OF_CONFIGURATIONS} configurations {timestamp}",
    compute=compute.to_dict(),
)
job_id = job["_id"]
print(f"✅ Created job: {job_id}")

## 7. Submit the job and monitor the status

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs

submit_jobs(client.jobs, [job_id])

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Results
### 8.1. Configurations ranked by final energy

`afir_function_eV` is the MACE energy plus the AFIR term at the end of the first relaxation, `energy_eV` the MACE
energy of the final structure; the `max_force` columns give the largest force on an atom at the end of each. The
`rotation_*` columns give the rotation applied to the molecule about its center, `position_u` and `position_v` the
fractional point of the surface cell it was placed over, and `molecule_fragments` how many pieces the molecule
ends in: 1 if intact, more if it split.

In [ ]:
import pandas as pd

file_url_by_name = {file["key"].rsplit("/", 1)[-1]: file["signedUrl"] for file in client.jobs.list_files(job_id)}
pd.read_csv(io.StringIO(await read_from_url(file_url_by_name["ranked_configurations.csv"])))

### 8.2. Final energy of each configuration

In [ ]:
from IPython.display import Image

Image(data=await read_from_url(file_url_by_name["final_energies.png"], as_bytes=True))

### 8.3. The lowest final structures

In [ ]:
from mat3ra.prode import PropertyName

final_structure_properties = client.properties.get_for_job(
    job_id, property_name=PropertyName.non_scalar.final_structure.value
)
visualize([
    {"material": Material.create(client.materials.get(final_structure_property["materialId"])), "title": f"Rank {rank}"}
    for rank, final_structure_property in enumerate(final_structure_properties, start=1)
])

## References

[1] S. Maeda, Y. Harabuchi, M. Takagi, K. Saita, K. Suzuki, T. Ichino, Y. Sumiya, K. Sugiyama and Y. Ono,
"Implementation and performance of the artificial force induced reaction method in the GRRM17 program",
J. Comput. Chem. 39, 233 (2018). https://doi.org/10.1002/jcc.25106

[2] S. Maeda and K. Morokuma, "Communications: A systematic method for locating transition structures of A + B → X
type reactions", J. Chem. Phys. 132, 241102 (2010). https://doi.org/10.1063/1.3457903

[3] K. Sugiyama, Y. Sumiya, M. Takagi, K. Saita and S. Maeda, "Understanding CO oxidation on the Pt(111) surface
based on a reaction route network", Phys. Chem. Chem. Phys. 21, 14366 (2019). https://doi.org/10.1039/c8cp06856a